In [2]:
spark.sql("show tables in spark_db").show()

+---------+-----------------+-----------+
|namespace|        tableName|isTemporary|
+---------+-----------------+-----------+
| spark_db|         diamonds|      false|
| spark_db|flight_time_clean|      false|
| spark_db|  flight_time_raw|      false|
| spark_db|    sf_fire_calls|      false|
+---------+-----------------+-----------+



In [3]:
flight_time_df = spark.read.table("spark_db.flight_time_clean")
flight_time_df.show()

+----------+----------+-----------------+------+----------------+----+--------------+--------------------+--------------------+--------------------+-------+--------------------+--------------------+---------+--------+--------------------+
|   FL_DATE|OP_CARRIER|OP_CARRIER_FL_NUM|ORIGIN|ORIGIN_CITY_NAME|DEST|DEST_CITY_NAME|        CRS_DEP_TIME|            DEP_TIME|           WHEELS_ON|TAXI_IN|        CRS_ARR_TIME|            ARR_TIME|CANCELLED|DISTANCE|              TAX_IN|
+----------+----------+-----------------+------+----------------+----+--------------+--------------------+--------------------+--------------------+-------+--------------------+--------------------+---------+--------+--------------------+
|2000-01-01|        DL|             1451|   BOS|      Boston, MA| ATL|   Atlanta, GA|INTERVAL '11:15' ...|INTERVAL '11:13' ...|INTERVAL '13:43' ...|      5|INTERVAL '14:00' ...|INTERVAL '13:48' ...|        0|     946|INTERVAL '05' MINUTE|
|2000-01-01|        DL|             1479|   

In [25]:
"""
Find the top 3 most delayed flights on 2000-01-16 from AUS to ORD
Collect the results into a list and display the output as the following.

    AA flight delayed by 5.0 minutes
    AA flight delayed by 2.0 minutes
    UA flight delayed by 2.0 minutes
"""

from pyspark.sql.functions import expr, col

top_3_df = (
    flight_time_df.filter((col("FL_DATE") == '2000-01-16') & (col("origin") == 'AUS') & (col("dest") == 'ORD'))\
                .withColumn("delayed_arrival", col('arr_time') - col('crs_arr_time'))\
                .orderBy("delayed_arrival", ascending = False)\
                .limit(3)
)

top_3_list_of_rows = top_3_df.collect()
# print(top_3_list_of_rows)

top_3_dict = [row.asDict() for row in top_3_list_of_rows] # asDict is a python function that converts structured data object into a dictionary 
# print(top_3_dict)

for i in top_3_dict:
    print(f"{i["OP_CARRIER"]} flight delayed by {i["delayed_arrival"].total_seconds() / 60} minutes")

AA flight delayed by 5.0 minutes
AA flight delayed by 2.0 minutes
UA flight delayed by 2.0 minutes


In [30]:
"""
Select only the 3rd most delayed flight
"""
top_3rd_df = (
    flight_time_df.filter((col("FL_DATE") == '2000-01-16') & (col("origin") == 'AUS') & (col("dest") == 'ORD'))\
                .withColumn("delayed_arrival", col('arr_time') - col('crs_arr_time'))\
                .orderBy("delayed_arrival", ascending = False)\
                .limit(3)\
                .offset(2) # this offsets by 2 giving only the 3rd row in the limit
)

top_3rd_df.show()

+----------+----------+-----------------+------+----------------+----+--------------+--------------------+--------------------+--------------------+-------+--------------------+--------------------+---------+--------+--------------------+--------------------+
|   FL_DATE|OP_CARRIER|OP_CARRIER_FL_NUM|ORIGIN|ORIGIN_CITY_NAME|DEST|DEST_CITY_NAME|        CRS_DEP_TIME|            DEP_TIME|           WHEELS_ON|TAXI_IN|        CRS_ARR_TIME|            ARR_TIME|CANCELLED|DISTANCE|              TAX_IN|     delayed_arrival|
+----------+----------+-----------------+------+----------------+----+--------------+--------------------+--------------------+--------------------+-------+--------------------+--------------------+---------+--------+--------------------+--------------------+
|2000-01-16|        UA|             1030|   AUS|      Austin, TX| ORD|   Chicago, IL|INTERVAL '07:20' ...|INTERVAL '07:20' ...|INTERVAL '09:40' ...|     10|INTERVAL '09:48' ...|INTERVAL '09:50' ...|        0|     978|INT